<a id="outputs-and-operations"></a>
# VideoDB Understanding: Outputs and Operations

Manage run lifecycle, analyzer status, output retrieval, validation failures, and cleanup.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/outputs-and-operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install, connect, and choose a video

In [ ]:
!pip install -q videodb python-dotenv

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()
print("Connected to VideoDB")
print("Collection:", collection.id)

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

video = collection.upload(VIDEO_URL)

# To use an existing video instead:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Video:", video.id)
video.play()

<a id="create"></a>
## 2. Create and inspect immediately

Creation returns an ID and initial status. Analyzer work continues asynchronously.

In [ ]:
understanding = video.understand(
    analyzers=[{"type": "spoken_words", "name": "transcript", "config": {"language": "en"}}],
    segmentation={"type": "time", "seconds": 30},
)

print("Understanding created")
print(f"ID: {understanding.id}")
print(f"Status: {understanding.status}")
print(f"Complete: {understanding.is_complete}")

<a id="refresh"></a>
## 3. Refresh status

`refresh()` updates the same object. Use it for custom polling or dashboards.

In [ ]:
understanding.refresh()
print(f"Run status: {understanding.status}")

print("\nAnalyzers:")
for analyzer in understanding.list_analyzers():
    print(f"- {analyzer.name}: {analyzer.status} (complete: {analyzer.is_complete})")

<a id="wait"></a>
## 4. Wait for a run or one analyzer

In [ ]:
transcript_analyzer = understanding.get_analyzer("transcript", refresh=True)
transcript_analyzer.wait_until_complete(timeout=1800, poll_interval=10)
understanding.wait_until_complete(timeout=1800, poll_interval=10)

print("Completion status:")
print(f"- Analyzer: {transcript_analyzer.status}")
print(f"- Understanding: {understanding.status}")

<a id="output"></a>
## 5. Fetch output only after success

In [ ]:
transcript_analyzer.refresh()
if transcript_analyzer.is_successful:
    output = transcript_analyzer.get_output()
    print(f"Scenes: {len(output.get('scenes', []))}")
else:
    print(f"Analyzer did not complete successfully: {transcript_analyzer.status}")

<a id="resume"></a>
## 6. Resume and list existing runs

In [ ]:
same_run = video.get_understanding(understanding.id)
print("Fetched Understanding")
print(f"ID: {same_run.id}")
print(f"Status: {same_run.status}")

print("\nAvailable Understandings:")
for run in video.list_understandings():
    print(f"- {run.id}: {run.status}")

<a id="validation"></a>
## 7. Validation failures

Invalid analyzer types, unsupported fields, bad sampling, missing dependencies, duplicate names, and cycles are rejected synchronously before a run is stored.

In [ ]:
RUN_VALIDATION_EXAMPLE = False

if RUN_VALIDATION_EXAMPLE:
    try:
        video.understand(
            analyzers=[{
                "type": "vlm",
                "name": "scene",
                "sampling": {"strategy": "interval", "every": 0},
                "config": {"prompt": "Describe the scene"},
            }]
        )
    except Exception as error:
        print(f"{type(error).__name__}: {error}")

## Status checklist

- `done`: output can be fetched.
- `failed`: inspect the analyzer/run error returned by the API.
- `skipped` or `cancelled`: terminal, but no successful output.
- A run may finish with a failed analyzer; inspect analyzers individually.
- `list_understandings()` is for discovery; fetch a specific run for current analyzer details.

## 8. Cleanup

In [ ]:
DELETE_UNDERSTANDING = False
if DELETE_UNDERSTANDING:
    understanding.delete()
    print("Deleted", understanding.id)